# Person 3 - Malak: Data Integration and RQ5-RQ6

Both datasets are aggregated independently by `zip_code` + `year` before a validated one-to-one merge. The cleaned snapshots supplied by the team are the reproducible inputs.


In [1]:
import pandas as pd

df_housing_clean = pd.read_csv("df_housing_clean.csv", dtype={"zip_code": "string"})
df_transport_clean = pd.read_csv("df_transport_clean.csv", dtype={"zip_code": "string"})
print("Housing:", df_housing_clean.shape)
print("Transportation:", df_transport_clean.shape)


Housing: (9551, 42)
Transportation: (8072, 26)


In [2]:
from scripts.integration import integrate

result = integrate(df_housing_clean, df_transport_clean)
df_housing_agg = result.df_housing_agg
df_transport_agg = result.df_transport_agg
merge_audit = result.merge_audit
df_integrated = result.df_integrated
print("Housing aggregated:", df_housing_agg.shape)
print("Transportation aggregated:", df_transport_agg.shape)


Housing aggregated: (1155, 7)
Transportation aggregated: (829, 7)


In [3]:
print("Merge results:")
print(merge_audit["_merge"].value_counts())
print("Final matched shape:", df_integrated.shape)


Merge results:
_merge
left_only     671
both          484
right_only    345
Name: count, dtype: int64
Final matched shape: (484, 11)


In [4]:
display(result.borough_disagreements)


,zip_code,year,housing_borough,transport_borough
606,10463,2024,Bronx / Manhattan,Bronx
607,10463,2025,Bronx / Manhattan,Bronx
1131,11237,2023,Brooklyn,Brooklyn / Queens
1134,11237,2026,Brooklyn,Brooklyn / Queens
1313,11385,2022,Queens,Brooklyn / Queens
1314,11385,2023,Queens,Brooklyn / Queens


In [5]:
print("Borough policy:", result.summary["borough_policy"])
print(df_integrated["borough"].value_counts())


Borough policy: most frequent housing borough per ZIP across the source snapshot; alphabetical tie break
borough
Brooklyn         153
Manhattan        111
Queens           106
Bronx            102
Staten Island     12
Name: count, dtype: Int64


In [6]:
print("Missing output values:")
print(df_integrated.isna().sum())
print("Duplicate ZIP-year keys:", df_integrated.duplicated(["zip_code", "year"]).sum())
print("Unknown affordable-unit source rows in matched groups:", result.summary["unknown_affordable_unit_rows_in_matched_groups"])
# The canonical integration function raises errors on invalid keys/counts,
# duplicate crash IDs, and missing injury counts before constructing the output.


Missing output values:
zip_code                  0
year                      0
housing_building_count    0
housing_project_count     0
affordable_units          0
total_housing_units       0
serious_crash_count       0
severe_injuries           0
non_severe_injuries       0
fatalities                0
borough                   0
dtype: int64
Duplicate ZIP-year keys: 0
Unknown affordable-unit source rows in matched groups: 3


In [7]:
df_integrated.to_csv("df_integrated.csv", index=False)
print("Saved df_integrated.csv:", df_integrated.shape)
print("For source aggregates and full audit evidence run: python scripts/integration.py --write")


Saved df_integrated.csv:

 (484, 11)
For source aggregates and full audit evidence run: python scripts/integration.py --write


### Method, audit and limitations

- The source snapshots have 9,551 housing rows and 8,072 transportation rows. Missing keys exclude 1,958 housing records (20.5% of rows, 1.044% of observed affordable units).
- Aggregate each source independently by `zip_code` and `year`, then perform an outer join to audit coverage. Keep the 484 matched ZIP-years as `df_integrated`; 671 housing-only and 345 transportation-only groups remain excluded. Unmatched activity is unknown for the counterpart dataset, so it is never filled with zero.
- Six matched ZIP-years have differing borough summaries. For reproducibility, the dashboard borough is the most frequent housing borough per ZIP across the source snapshot, with alphabetical tie breaking. This analytical convention does not verify ZIP boundaries or correct source geography.
- All count columns are validated as nonnegative integers when known. Crash IDs must be unique before injury sums are computed. The output has 11 columns, no missing values and unique ZIP-year keys for the supplied snapshots.
- Housing contains 29 unknown affordable-unit values, three in matched groups. Observed sums may undercount. An entirely unknown group stays missing through `sum(min_count=1)` rather than becoming zero. No source values are imputed.
- The 44 source transportation records with ZIP `10000` are retained in the source audit for review; none match the housing snapshot.
- These results describe matched ZIP-years with available geography. The cleaned transportation dates run from 2022-10-01 to 2026-06-30; missing source ZIPs and inner matching also affect coverage. Calendar years 2023-2025 avoid the boundary dates but are still an incomplete selected sample, not complete NYC totals.
- Housing includes preservation and new construction. `year` is project start year, not completion year. Summing annual building/project counts counts ZIP-year appearances, not unique buildings/projects across the entire period.
- Findings describe associations; housing production is not a denominator for traffic exposure and these data cannot establish causal effects.

Detailed audit evidence is generated by `python scripts/integration.py --write`. See `docs/INTEGRATION_METHOD.md`.


In [8]:
import plotly.express as px

rq1 = df_integrated[df_integrated["year"].between(2023, 2025)].copy()

housing_limit = rq1["affordable_units"].quantile(0.75)
crash_limit = rq1["serious_crash_count"].quantile(0.75)

rq1["high_both"] = (
    (rq1["affordable_units"] >= housing_limit) &
    (rq1["serious_crash_count"] >= crash_limit)
)

rq1["category"] = rq1["high_both"].map({
    True: "High in both",
    False: "Other"
})

fig = px.scatter(
    rq1,
    x="affordable_units",
    y="serious_crash_count",
    color="category",
    hover_data=["zip_code", "year", "borough"],
    title="Affordable Housing vs Serious Crashes (2023–2025)",
    labels={
        "affordable_units": "Affordable Housing Units",
        "serious_crash_count": "Serious Crash Count",
        "category": "Category"
    }
)

fig.add_vline(x=housing_limit, line_dash="dash")
fig.add_hline(y=crash_limit, line_dash="dash")
fig.show()

print("ZIP-year records with high housing and high crashes:")
display(
    rq1.loc[rq1["high_both"],
    ["zip_code", "year", "borough", "affordable_units", "serious_crash_count"]]
    .sort_values("serious_crash_count", ascending=False)
    .head(10)
)

ZIP-year records with high housing and high crashes:


,zip_code,year,borough,affordable_units,serious_crash_count
365,11212,2025,Brooklyn,1090.0,43.0
335,10456,2025,Bronx,923.0,36.0
146,11207,2023,Brooklyn,702.0,36.0
131,10468,2023,Bronx,516.0,32.0
121,10456,2023,Bronx,712.0,30.0
361,11208,2025,Brooklyn,612.0,30.0
228,10457,2024,Bronx,1105.0,28.0
236,10467,2024,Bronx,753.0,27.0
332,10453,2025,Bronx,522.0,27.0
373,11220,2025,Brooklyn,811.0,27.0


**Finding:** Several ZIP codes in Brooklyn and the Bronx showed both high affordable housing production and high serious-crash counts during 2023–2025. For example, ZIP code 11212 (Brooklyn) recorded 1,090 affordable units and 43 serious crashes in 2025. This suggests overlapping housing and road-safety concerns, but does not establish causation.

In [9]:
rq2 = (
    df_integrated[df_integrated["year"].between(2023, 2025)]
    .groupby(["zip_code", "borough"], as_index=False)
    .agg(
        affordable_units=("affordable_units", "sum"),
        serious_crash_count=("serious_crash_count", "sum"),
        years_observed=("year", "nunique")
    )
)

rq2 = rq2[
    (rq2["years_observed"] == 3) &
    (rq2["affordable_units"] >= 100)
].copy()

rq2["crashes_per_100_units"] = (
    rq2["serious_crash_count"] / rq2["affordable_units"] * 100
)

top10 = rq2.nlargest(10, "crashes_per_100_units")

fig = px.bar(
    top10.sort_values("crashes_per_100_units"),
    x="crashes_per_100_units",
    y="zip_code",
    color="borough",
    orientation="h",
    hover_data=["affordable_units", "serious_crash_count"],
    title="Top 10 ZIP Codes: Serious Crashes Relative to Affordable Units",
    labels={
        "crashes_per_100_units": "Crashes per 100 Affordable Units",
        "zip_code": "ZIP Code"
    }
)

fig.show()

display(top10[
    ["zip_code", "borough", "affordable_units",
     "serious_crash_count", "crashes_per_100_units"]
].round(2))

,zip_code,borough,affordable_units,serious_crash_count,crashes_per_100_units
99,11234,Brooklyn,157.0,103.0,65.61
119,11377,Queens,103.0,67.0,65.05
54,10466,Bronx,135.0,62.0,45.93
127,11432,Queens,149.0,64.0,42.95
90,11223,Brooklyn,139.0,55.0,39.57
73,11206,Brooklyn,396.0,89.0,22.47
23,10032,Manhattan,195.0,42.0,21.54
102,11237,Brooklyn,198.0,39.0,19.70
52,10462,Bronx,274.0,47.0,17.15
105,11249,Brooklyn,167.0,27.0,16.17


**Finding:** ZIP codes 11234 (Brooklyn) and 11377 (Queens) had the highest serious-crash counts relative to affordable housing production during 2023–2025. This comparison helps identify areas worth further investigation, but it does not represent actual crash risk because housing units are not a measure of traffic exposure.

## Reproduce the integration

Run `python scripts/integration.py` to compare the committed CSV with a fresh transformation without modifying files. Run `python scripts/integration.py --write` to regenerate both source aggregates, the final CSV, unmatched groups, borough disagreements and the JSON audit. Run `python -m unittest discover -s tests -v` for the exact-snapshot and synthetic many-to-many, unknown-unit and validation regression checks.

RQ5 uses individual matched ZIP-year records from calendar years 2023-2025; the dashed lines are the sample 75th percentiles. RQ6 ranks only ZIPs observed in all three selected years and with at least 100 observed affordable units. These inclusion rules should remain visible when presenting either chart.
